In [13]:
import csv
import json

csv_path = "/home/lauosgom/anomaly/llamatel-crisis-lifeline-chatbot/telesperanza_faq.csv"      # change this to your file
output_path = "/home/lauosgom/anomaly/llamatel-crisis-lifeline-chatbot/faq_data.json"

faqs = []
with open(csv_path, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        faqs.append({
            "question": row["question"].strip(),
            "answer": row["answer"].strip(),
        })

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(faqs, f, indent=2, ensure_ascii=False)

print(f"Wrote {len(faqs)} FAQ entries to {output_path}")

Wrote 61 FAQ entries to /home/lauosgom/anomaly/llamatel-crisis-lifeline-chatbot/faq_data.json


In [12]:
with open(csv_path, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row["question"].strip())

¿Cuál es el nombre legal de la organización?
¿Cuál es el NIT de la organización?
¿El Teléfono de la Esperanza es una empresa con ánimo de lucro?
¿Tienen alguna afiliación política o religiosa?
¿Cuál es la misión principal del Teléfono de la Esperanza?
¿Cuál es el propósito o visión frente a los problemas?
¿Quién vigila o inspecciona a la asociación?
¿A qué gremio pertenecen?
¿Cuánto tiempo llevan prestando servicio?
¿Han recibido algún reconocimiento o premio?
¿A qué números puedo llamar si necesito ayuda?
¿En qué horario atienden las líneas?
¿La orientación telefónica tiene algún costo?
¿Tengo que decir mi nombre cuando llamo?
¿Qué tipo de crisis atienden en la línea?
¿Quién contesta mis llamadas?
¿La ayuda es inmediata o debo pedir cita?
¿Cuál es la metodología de ayuda?
¿Atienden urgencias psiquiátricas graves?
¿Cuáles son las palabras clave de la labor del orientador?
¿Ofrecen asesoría con profesionales?
¿La asesoría profesional es presencial o telefónica?
¿Qué pasa si no tienen un

In [5]:
import pandas as pd
data = pd.read_csv(csv_path)
data.head()

,question,answer
0,¿Cuál es el nombre legal de la organización?,El nombre legal es Asociación Colombiana del T...
1,¿Cuál es el NIT de la organización?,El NIT de la Asociación es 830022458-5.
2,¿El Teléfono de la Esperanza es una empresa co...,"No, somos una entidad sin ánimo de lucro (ONG)..."
3,¿Tienen alguna afiliación política o religiosa?,"No, somos una organización aconfesional y apol..."
4,¿Cuál es la misión principal del Teléfono de l...,Nuestra misión es promover y prevenir la salud...


In [2]:
"""
build_index.py
Reads faq_data.json, embeds each entry with OpenAI, and loads the result
into a BigQuery table. Run this once whenever your FAQ document changes
(it truncates and reloads the table each time, so it's safe to re-run).

Requires:
  - A .env file (copy .env.example to .env and fill in) with your
    OPENAI_API_KEY, GCP_PROJECT_ID, and optionally
    GOOGLE_APPLICATION_CREDENTIALS if not using ADC
  - A GCP project with the BigQuery API enabled
  - A dataset already created (see README) — this script creates the
    table itself if it doesn't exist
"""

import json
import os

from dotenv import load_dotenv
from google.cloud import bigquery
from openai import OpenAI

ModuleNotFoundError: No module named 'openai'

In [ ]:


load_dotenv()

FAQ_JSON_PATH = "faq_data.json"
EMBEDDING_MODEL = "text-embedding-3-small"

PROJECT_ID = os.environ["GCP_PROJECT_ID"]
DATASET_ID = os.environ.get("BQ_DATASET_ID", "faq_bot")
TABLE_ID = os.environ.get("BQ_TABLE_ID", "faq_embeddings")
TABLE_FQN = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
bq_client = bigquery.Client(project=PROJECT_ID)

SCHEMA = [
    bigquery.SchemaField("id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("question", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("answer", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("embedding", "FLOAT64", mode="REPEATED"),
]


def embed_texts(texts: list[str]) -> list[list[float]]:
    resp = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [d.embedding for d in resp.data]


def ensure_table_exists():
    try:
        bq_client.get_table(TABLE_FQN)
    except Exception:
        table = bigquery.Table(TABLE_FQN, schema=SCHEMA)
        bq_client.create_table(table)
        print(f"Created table {TABLE_FQN}")


def main():
    with open(FAQ_JSON_PATH, "r", encoding="utf-8") as f:
        faqs = json.load(f)

    # Each FAQ entry is its own chunk — no need to split further.
    texts_to_embed = [f"Q: {item['question']}\nA: {item['answer']}" for item in faqs]

    print(f"Embedding {len(texts_to_embed)} FAQ entries...")
    embeddings = embed_texts(texts_to_embed)

    rows = [
        {
            "id": f"faq-{i}",
            "question": item["question"],
            "answer": item["answer"],
            "embedding": embedding,
        }
        for i, (item, embedding) in enumerate(zip(faqs, embeddings))
    ]

    ensure_table_exists()

    # Truncate + reload so re-running this script never leaves stale/duplicate rows.
    bq_client.query(f"TRUNCATE TABLE `{TABLE_FQN}`").result()

    job_config = bigquery.LoadJobConfig(
        schema=SCHEMA,
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    )
    load_job = bq_client.load_table_from_json(rows, TABLE_FQN, job_config=job_config)
    load_job.result()  # wait for completion

    print(f"Loaded {len(rows)} FAQ entries into {TABLE_FQN}")


if __name__ == "__main__":
    main()